## 06b_lenormand_skmob_pl
  Same as 06 for PL NUTS3, in two variants for the unpublished small flows.
  Output: data/interim/lenormand_comparison/pl/, metrics CSV. Environment: nuts3-mobility.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

sys.path.append(str(Path.cwd().parent))  # notebook is in notebooks/
from src import config

# ----------------------------- CONFIG -----------------------------
COUNTRY = "PL"
YEAR = 2023
COL_ID, COL_POP = "nuts3_id", "population"
COL_COUNTRY, COL_YEAR = "country_code", "year"
NUTS3_TABLE = config.PROCESSED_DIR / "nuts3_master_table.parquet"
SHARED_DIR = config.INTERIM_DIR / "lenormand_comparison" / COUNTRY.lower()
POP_CSV = SHARED_DIR / "population.csv"
# ------------------------------------------------------------------

SHARED_DIR.mkdir(parents=True, exist_ok=True)

tbl = pd.read_parquet(NUTS3_TABLE, columns=[COL_ID, COL_COUNTRY, COL_YEAR, COL_POP])
pop = (tbl[(tbl[COL_COUNTRY] == COUNTRY) & (tbl[COL_YEAR] == YEAR)]
       [[COL_ID, COL_POP]]
       .sort_values(COL_ID)
       .reset_index(drop=True))
assert pop[COL_ID].is_unique, "duplicated region ids"
assert pop[COL_POP].notna().all(), "missing population values"

pop.to_csv(POP_CSV, index=False)
print("Regions:", len(pop))
print("Total population:", int(pop[COL_POP].sum()))
print("Saved:", os.path.relpath(POP_CSV))

In [ ]:
# PROJ data fix: must run BEFORE geopandas / pyproj / skmob are imported
import os
import sys
from pathlib import Path
from importlib.util import find_spec


def find_proj_dir():
    """Return the first directory that contains proj.db (no pyproj import needed)."""
    candidates = [Path(sys.prefix) / "share" / "proj"]  # conda environment
    spec = find_spec("pyproj")
    if spec and spec.submodule_search_locations:  # pip wheel with bundled data
        pyproj_dir = Path(list(spec.submodule_search_locations)[0])
        candidates.append(pyproj_dir / "proj_dir" / "share" / "proj")
    for c in candidates:
        if (c / "proj.db").exists():
            return c
    return None


proj_dir = find_proj_dir()
if proj_dir is not None:
    os.environ["PROJ_LIB"] = str(proj_dir)
    os.environ["PROJ_DATA"] = str(proj_dir)
    print("PROJ data: proj.db found, environment variables set")
else:
    print("PROJ data: proj.db NOT found in the expected places")

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely.wkb
from geopandas.array import GeometryArray
from scipy.spatial.distance import pdist, squareform

# ----------------------------- CONFIG -----------------------------
CRS_GEO = "EPSG:4326"
CRS_METRIC = "EPSG:3035"
COL_GEOM = "geometry"
DIST_CSV = SHARED_DIR / "distance_km.csv"
# COUNTRY, YEAR, COL_ID, COL_COUNTRY, COL_YEAR, NUTS3_TABLE, SHARED_DIR, pop come from the previous cell
# ------------------------------------------------------------------

tbl = pd.read_parquet(NUTS3_TABLE, columns=[COL_ID, COL_COUNTRY, COL_YEAR, COL_GEOM])
tbl = (tbl[(tbl[COL_COUNTRY] == COUNTRY) & (tbl[COL_YEAR] == YEAR)]
       .sort_values(COL_ID).reset_index(drop=True))
assert tbl[COL_ID].tolist() == pop[COL_ID].tolist(), "region order differs from population.csv"

# Geometry is stored as WKB bytes; element-wise assignment (vectorized fails with this shapely version)
geoms = tbl[COL_GEOM].apply(
    lambda x: shapely.wkb.loads(x) if isinstance(x, (bytes, bytearray)) else x).tolist()
geom_arr = np.empty(len(geoms), dtype=object)
for i, g in enumerate(geoms):
    geom_arr[i] = g
regions = gpd.GeoDataFrame(tbl.drop(columns=COL_GEOM), geometry=GeometryArray(geom_arr, crs=CRS_GEO))

# Centroids in a metric projection, pairwise distances in km
regions_metric = regions.to_crs(CRS_METRIC)
coords = np.array([[p.x, p.y] for p in regions_metric.geometry.centroid])
dist_km = squareform(pdist(coords)) / 1000

ids = regions[COL_ID].tolist()
dist_df = pd.DataFrame(dist_km, index=ids, columns=ids)
dist_df.to_csv(DIST_CSV)

offdiag = ~np.eye(len(ids), dtype=bool)
print("Distance matrix:", dist_df.shape)
print("Min off-diagonal distance (km):", round(float(dist_km[offdiag].min()), 1))
print("Max distance (km):", round(float(dist_km.max()), 1))
print("Saved:", os.path.relpath(DIST_CSV))

In [ ]:
# ----------------------------- CONFIG -----------------------------
FLOWS_CSV = config.INTERIM_DIR / "05b_gravity_pl" / "PL_nuts3_flows.csv"
COL_FLOW_ORIGIN, COL_FLOW_DEST, COL_FLOW = "origin", "destination", "flow"
MISSING_VALUES = {"missing0": 0, "missing2": 2}      # value given to the unobserved (T < 3) pairs
OD_CSV_TEMPLATE = "od_observed_{variant}.csv"
# SHARED_DIR, ids come from the previous cells
# ------------------------------------------------------------------

flows_raw = pd.read_csv(FLOWS_CSV)
dup_pairs = int(flows_raw.duplicated(subset=[COL_FLOW_ORIGIN, COL_FLOW_DEST]).sum())
assert dup_pairs == 0, "duplicated origin-destination pairs"
unknown_ids = sorted((set(flows_raw[COL_FLOW_ORIGIN]) | set(flows_raw[COL_FLOW_DEST])) - set(ids))
assert not unknown_ids, "flow file contains ids that are not among the regions"

# Long table -> matrix in the region order; unobserved pairs are NaN for now
od_obs = (flows_raw
          .pivot(index=COL_FLOW_ORIGIN, columns=COL_FLOW_DEST, values=COL_FLOW)
          .reindex(index=ids, columns=ids))
n = len(ids)
offdiag = ~np.eye(n, dtype=bool)
observed_mask = od_obs.notna().to_numpy() & offdiag
missing_mask = od_obs.isna().to_numpy() & offdiag
assert not od_obs.to_numpy()[~offdiag][~np.isnan(od_obs.to_numpy()[~offdiag])].any(), "non-zero diagonal"

print("Regions:", n, "| directed pairs:", n * (n - 1))
print("Observed pairs (T >= 3):", int(observed_mask.sum()),
      "| missing pairs:", int(missing_mask.sum()),
      "| observed share:", round(float(observed_mask.sum() / (n * (n - 1))), 3))

for variant, fill in MISSING_VALUES.items():
    mat = od_obs.to_numpy(dtype=float).copy()
    mat[np.isnan(mat)] = fill
    np.fill_diagonal(mat, 0)                          # T_ii = 0
    out_trips = mat.sum(axis=1)
    assert (out_trips > 0).all(), "a region has zero outflow"
    path = SHARED_DIR / OD_CSV_TEMPLATE.format(variant=variant)
    pd.DataFrame(mat, index=ids, columns=ids).to_csv(path)
    print(f"{variant}: total flow {int(mat.sum())} | smallest outflow {int(out_trips.min())} "
          f"| largest {int(out_trips.max())} | saved: {os.path.relpath(path)}")

In [ ]:
import skmob
import geopandas as gpd

# ----------------------------- CONFIG -----------------------------
TILE_ID = "tile_ID"
OUTFLOW_COL = "tot_outflow"
COL_FLOW_ORIGIN, COL_FLOW_DEST, COL_FLOW = "origin", "destination", "flow"
VARIANTS = list(MISSING_VALUES)                       # e.g. missing0, missing2
# ids, pop, regions, SHARED_DIR, OD_CSV_TEMPLATE, COL_ID, COL_POP, COL_GEOM, CRS_GEO
# come from the earlier cells
# ------------------------------------------------------------------

# Base tessellation: one row per region (id, geometry, population), in the shared order
base = regions[[COL_ID, COL_GEOM]].merge(pop, on=COL_ID).rename(columns={COL_ID: TILE_ID})
base = gpd.GeoDataFrame(base, geometry=COL_GEOM, crs=CRS_GEO)
assert base[TILE_ID].tolist() == ids, "region order differs from the population table"

tessellations, observed_offdiag, flows_long, fdfs = {}, {}, {}, {}
for variant in VARIANTS:
    od = pd.read_csv(SHARED_DIR / OD_CSV_TEMPLATE.format(variant=variant), index_col=0)
    assert list(od.index) == ids == list(od.columns), "OD matrix order differs"
    mat = od.to_numpy(dtype=float)
    assert np.trace(mat) == 0, "the diagonal is not empty"
    observed_offdiag[variant] = mat

    # Tessellation of this variant: total outflow of each region (row sum)
    tess = base.copy()
    tess[OUTFLOW_COL] = mat.sum(axis=1)
    tessellations[variant] = tess

    # Long format: one row per origin-destination pair (diagonal included);
    # axis names are set explicitly because rows and columns carry the same name
    flows_long[variant] = (od.rename_axis(index=COL_FLOW_ORIGIN, columns=COL_FLOW_DEST)
                           .stack().rename(COL_FLOW).reset_index())
    fdfs[variant] = skmob.FlowDataFrame(
        flows_long[variant], origin=COL_FLOW_ORIGIN, destination=COL_FLOW_DEST,
        flow=COL_FLOW, tessellation=tess, tile_id=TILE_ID)

    print(f"{variant}: tiles {len(tess)} | flow rows {len(fdfs[variant])} "
          f"(expected {len(ids) ** 2}) | total flow {int(fdfs[variant][COL_FLOW].sum())} "
          f"| outflow min {int(tess[OUTFLOW_COL].min())}, max {int(tess[OUTFLOW_COL].max())}")

In [ ]:
import warnings
from skmob.models.gravity import Gravity

# ----------------------------- CONFIG -----------------------------
GRAVITY_LAWS = {"gravity_power": "power_law", "gravity_exp": "exponential"}
CONSTRAINT_LEVELS = {"UM": "globally constrained", "PCM": "singly constrained"}
RESULTS_DIR = config.PROCESSED_DIR / "06b_lenormand_skmob_pl"
GRAVITY_PARAMS_CSV = RESULTS_DIR / "gravity_params.csv"
# VARIANTS, flows_long, tessellations, COL_*, TILE_ID, COL_POP come from the earlier cells
# ------------------------------------------------------------------

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if GRAVITY_PARAMS_CSV.exists():
    gravity_params = pd.read_csv(GRAVITY_PARAMS_CSV)
    print("Loaded from checkpoint:", os.path.relpath(GRAVITY_PARAMS_CSV))
else:
    rows = []
    for variant in VARIANTS:
        # Fit on the off-diagonal flows only (T_ii = 0, as in the paper)
        fl = flows_long[variant]
        fdf_off = skmob.FlowDataFrame(
            fl[fl[COL_FLOW_ORIGIN] != fl[COL_FLOW_DEST]],
            origin=COL_FLOW_ORIGIN, destination=COL_FLOW_DEST, flow=COL_FLOW,
            tessellation=tessellations[variant], tile_id=TILE_ID)
        for law, deterrence in GRAVITY_LAWS.items():
            for level, gravity_type in CONSTRAINT_LEVELS.items():
                row = {"variant": variant, "model": f"{law}_{level}",
                       "deterrence_func_type": deterrence, "gravity_type": gravity_type}
                # Record warnings instead of silencing them; only the class names are kept
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter("always")
                    try:
                        model = Gravity(gravity_type=gravity_type, deterrence_func_type=deterrence)
                        model.fit(fdf_off, relevance_column=COL_POP)
                        row.update(gamma=model.deterrence_func_args[0],
                                   origin_exp=model.origin_exp,
                                   destination_exp=model.destination_exp,
                                   fit_error="")
                    except Exception as e:               # keep only the exception class name
                        row.update(gamma=np.nan, origin_exp=np.nan, destination_exp=np.nan,
                                   fit_error=type(e).__name__)
                row["fit_warnings"] = ";".join(sorted({w.category.__name__ for w in caught}))
                rows.append(row)
    gravity_params = pd.DataFrame(rows)
    gravity_params.to_csv(GRAVITY_PARAMS_CSV, index=False)
    print("Saved:", os.path.relpath(GRAVITY_PARAMS_CSV))

# Model objects from the saved parameters (only the successful fits)
ok = gravity_params[gravity_params["fit_error"].fillna("") == ""]
gravity_models = {
    (r.variant, r.model): Gravity(deterrence_func_type=r.deterrence_func_type,
                                  deterrence_func_args=[r.gamma],
                                  origin_exp=r.origin_exp,
                                  destination_exp=r.destination_exp,
                                  gravity_type=r.gravity_type)
    for r in ok.itertuples()
}

print(gravity_params[["variant", "model", "gamma", "origin_exp", "destination_exp",
                      "fit_error", "fit_warnings"]].round(6).to_string(index=False))

In [ ]:
from skmob.models.radiation import Radiation

# ----------------------------- CONFIG -----------------------------
N_REPS = 100                        # stochastic realizations per model (as in the paper)
SEED_BASE = 0                       # realization i uses seed SEED_BASE + i
RADIATION_NAME = "radiation_PCM"
REALIZATION_PATH = lambda variant, name: RESULTS_DIR / f"realizations_{variant}_{name}.npy"
# VARIANTS, gravity_models, tessellations, observed_offdiag, ids, TILE_ID, OUTFLOW_COL,
# COL_POP, COL_FLOW_*, RESULTS_DIR come from the earlier cells
# ------------------------------------------------------------------

# Model objects per variant: successful gravity fits + radiation
model_objects = {}
for variant in VARIANTS:
    for (v, name), model in gravity_models.items():
        if v == variant:
            model_objects[(variant, name)] = model
    model_objects[(variant, RADIATION_NAME)] = Radiation()

realizations = {}
for (variant, name), model in model_objects.items():
    path = REALIZATION_PATH(variant, name)
    if path.exists():
        realizations[(variant, name)] = np.load(path)
        print(variant, name, "- loaded from checkpoint:", os.path.relpath(path))
        continue

    mats, warning_types = [], set()
    for i in range(N_REPS):
        np.random.seed(SEED_BASE + i)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            synth = model.generate(
                tessellations[variant],
                tile_id_column=TILE_ID,
                tot_outflows_column=OUTFLOW_COL,
                relevance_column=COL_POP,
                out_format="flows")
        warning_types |= {w.category.__name__ for w in caught}
        # FlowDataFrame -> plain DataFrame -> matrix in the shared region order
        mats.append(pd.DataFrame(synth)
                    .pivot(index=COL_FLOW_ORIGIN, columns=COL_FLOW_DEST, values=COL_FLOW)
                    .reindex(index=ids, columns=ids).fillna(0).to_numpy())
    realizations[(variant, name)] = np.stack(mats)      # shape: (N_REPS, n_regions, n_regions)
    np.save(path, realizations[(variant, name)])
    print(variant, name, "- generated and saved:", os.path.relpath(path),
          "| warnings:", ";".join(sorted(warning_types)) or "none")

print()
for (variant, name), arr in realizations.items():
    diag_sum = float(np.trace(arr, axis1=1, axis2=2).sum())
    print(f"{variant} {name}: shape {arr.shape} | diagonal sum {diag_sum:.0f} | "
          f"mean total {arr.sum(axis=(1, 2)).mean():.0f} "
          f"(observed {observed_offdiag[variant].sum():.0f})")

In [ ]:
import skmob.measures.evaluation as ev

# ----------------------------- CONFIG -----------------------------
METRICS_BY_REP_CSV = RESULTS_DIR / "metrics_skmob_by_rep.csv"
METRICS_SUMMARY_CSV = RESULTS_DIR / "metrics_skmob_summary.csv"
# realizations, observed_offdiag, ids, VARIANTS come from the earlier cells
# ------------------------------------------------------------------

# Off-diagonal pairs only (73 x 72 = 5256 pairs)
offdiag = ~np.eye(len(ids), dtype=bool)

if METRICS_BY_REP_CSV.exists():
    by_rep = pd.read_csv(METRICS_BY_REP_CSV)
    print("Loaded from checkpoint:", os.path.relpath(METRICS_BY_REP_CSV), by_rep.shape)
else:
    rows = []
    for (variant, name), arr in realizations.items():
        obs_flat = observed_offdiag[variant][offdiag]
        for rep, mat in enumerate(arr):
            sim_flat = mat[offdiag].astype(float)
            # information gain is infinite when a real pair gets zero predicted flow
            with np.errstate(divide="ignore", invalid="ignore"):
                ig = ev.information_gain(obs_flat, sim_flat)
            rows.append({
                "variant": variant,
                "model": name,
                "rep": rep,
                "cpc": ev.common_part_of_commuters(obs_flat, sim_flat),
                "information_gain": ig,
                "zero_pairs": int(((obs_flat > 0) & (sim_flat == 0)).sum()),
            })
    by_rep = pd.DataFrame(rows)
    by_rep.to_csv(METRICS_BY_REP_CSV, index=False)
    print("Saved:", os.path.relpath(METRICS_BY_REP_CSV), by_rep.shape)

# Summary per variant and model over the realizations
summary = (
    by_rep.groupby(["variant", "model"], sort=False)
    .agg(cpc_mean=("cpc", "mean"), cpc_std=("cpc", "std"),
         cpc_min=("cpc", "min"), cpc_max=("cpc", "max"),
         zero_pairs_mean=("zero_pairs", "mean"),
         n_ig_nonfinite=("information_gain", lambda s: int((~np.isfinite(s)).sum())),
         ig_mean_finite=("information_gain", lambda s: s[np.isfinite(s)].mean()))
    .reset_index()
)
summary.to_csv(METRICS_SUMMARY_CSV, index=False)
print("Saved:", os.path.relpath(METRICS_SUMMARY_CSV))
print(summary.round(4).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from pathlib import Path

# ----------------------------- CONFIG -----------------------------
FIG_NAME = "pl_lenormand_cpc_skmob.png"
FIG_DIRS = [Path("../figures"), Path("../plots")]
LAW_STYLE = {                        # color, marker, legend label
    "gravity_exp":   ("#2a78d6", "o", "Gravity (exp)"),
    "gravity_power": ("#eb6834", "s", "Gravity (power)"),
    "radiation":     ("#1baf7a", "^", "Radiation"),
}
LAW_Y_OFFSET = {"gravity_exp": 0.18, "gravity_power": 0.0, "radiation": -0.18}
LEVELS = ["UM", "PCM"]
X_PAD = 0.03
# summary, VARIANTS come from the earlier cells
# ------------------------------------------------------------------

s = summary.copy()
s["law"] = s["model"].str.rsplit("_", n=1).str[0]
s["level"] = s["model"].str.rsplit("_", n=1).str[1]
ypos = {lvl: len(LEVELS) - i for i, lvl in enumerate(LEVELS)}      # UM on top
xmin, xmax = s["cpc_mean"].min() - X_PAD, s["cpc_mean"].max() + X_PAD

fig, axes = plt.subplots(1, len(VARIANTS), figsize=(10, 3.8), sharey=True)
for ax, variant in zip(np.atleast_1d(axes), VARIANTS):
    for law, (color, marker, label) in LAW_STYLE.items():
        sub = s[(s["variant"] == variant) & (s["law"] == law)]
        ax.scatter(sub["cpc_mean"], [ypos[l] + LAW_Y_OFFSET[law] for l in sub["level"]],
                   color=color, marker=marker, s=60, edgecolor="white", linewidth=1.2,
                   label=label, zorder=3)
    ax.set_yticks(list(ypos.values()))
    ax.set_yticklabels(list(ypos.keys()))
    ax.set_xlim(xmin, xmax)
    ax.xaxis.set_major_locator(MultipleLocator(0.05))   # fewer ticks, so labels do not overlap
    ax.set_xlabel("CPC (mean of replications)")
    ax.set_title(f"Unobserved pairs (T < 3) set to {MISSING_VALUES[variant]}", fontweight="bold")
    ax.grid(axis="x", color="#e1e0d9", linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.tick_params(axis="y", length=0)

handles, labels = np.atleast_1d(axes)[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False)
fig.tight_layout(rect=(0, 0.1, 1, 1))

for d in FIG_DIRS:
    d.mkdir(parents=True, exist_ok=True)
    fig.savefig(d / FIG_NAME, dpi=200, facecolor="white")
    print("Saved:", os.path.relpath(d / FIG_NAME))
plt.show()